# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Content Opportunity Scoring (which pages to review first). **Data:** 30k-row starter slice, one row per page, trailing 90 days. **Target:** `declining = 1` when `trend_direction == "down"` (impressions in the last 30 days fell more than 20% vs the 30 days before). Wording below is observed / directional / decision-support.

In [1]:
import os, json, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"): os.chdir("../..")
os.makedirs("work/outputs", exist_ok=True)
PATH = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(PATH):
    import urllib.request
    PATH = "content_refresh_anonymized.csv"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Hamzaviour/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv", PATH)
df = pd.read_csv(PATH)
df["declining"] = (df.trend_direction == "down").astype(int)   # target, built from trend_direction
SEED = 42
print(df.shape, "base rate declining:", round(df.declining.mean(), 3), "| clients:", df.client_id.nunique())

(30000, 45) base rate declining: 0.542 | clients: 32


## 1. Method choice and why

The question is "which pages should be reviewed first?", so I need a **score that ranks**, evaluated at **precision@K** (of the K pages flagged, how many really are declining), next to the base rate. The target is yes/no and observed, so the skill's path is: Logistic Regression (readable) → a shallow Decision Tree (printable) → Random Forest (stronger, only kept if the table earns it). No Gradient Boosting: with 30k rows and mostly weak signals it would add complexity without a reason.

**Features (all from the 90-day snapshot):** search volume, competition, word count, log traffic totals, days with impressions/sessions, page age, days since update, CTR, position, engagement, content type, intent. Missing values get a `has_` flag instead of a blind zero-fill (missingness follows `content_type`).

**Excluded on purpose:** `trend_direction`, `trend_pct` (the label), `impressions_last_30d / prev_30d` and their click/session twins (the label is computed from them), IDs, and `provider_used / model_used`.

**Known caveat:** `impressions_90d` contains the last 30 days, so it overlaps the label window a little. It is a snapshot-level limit I cannot remove with this file; it is why I read the importances before believing any score.

In [2]:
EXCLUDED = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "content_id", "client_id",
            "provider_used", "model_used", "declining"]
NUM = ["search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d", "clicks_90d",
       "sessions_90d", "ai_sessions_90d", "days_with_impressions", "days_with_sessions", "content_age_days",
       "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate"]
CAT = ["content_type", "main_intent", "competition_level"]
assert not set(NUM + CAT) & set(EXCLUDED)

X = df[NUM + CAT].copy()
for c in ["search_volume", "word_count", "engagement_rate", "scroll_rate"]:
    X["has_" + c] = X[c].notna().astype(int)
X["pos_missing"] = (X.avg_position == 0).astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    X[c] = np.log1p(X[c])
X = pd.get_dummies(X, columns=CAT, dummy_na=True)
y, groups = df.declining.values, df.client_id.values
print("feature matrix:", X.shape)

feature matrix: (30000, 35)


## 2. Split design

**Grouped by client, 5-fold (`GroupKFold`).** Pages from the same client share a site, a template and a niche, so a random split would let the model memorise client habits and look better than it is. Here every client is entirely in train or entirely in test, which matches the real use ("score a client we have not seen"). Every number below is computed on held-out folds only. The baseline and every model use the **identical folds**, the same target and the same metrics. The baseline's expected-CTR norm is re-fitted on each training fold only. Seeds fixed at 42.

In [3]:
from sklearn.model_selection import GroupKFold
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: train {len(tr):>6,}  test {len(te):>6,}  test clients {len(set(groups[te])):>2}  test base rate {y[te].mean():.3f}")

fold 0: train 22,992  test  7,008  test clients  1  test base rate 0.490
fold 1: train 24,269  test  5,731  test clients  7  test base rate 0.645
fold 2: train 24,247  test  5,753  test clients  8  test base rate 0.379
fold 3: train 24,245  test  5,755  test clients  8  test base rate 0.622
fold 4: train 24,247  test  5,753  test clients  8  test base rate 0.585


## 3. Train + compare vs my baseline

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

def prec_at_k(score, label, k):
    return label[np.argsort(-score, kind="stable")[:k]].mean()

def baseline_score(tr, te):
    """Week-4 rule: est. missed clicks = impressions * max(0, expected CTR(position band) - CTR). Norm from TRAIN fold only."""
    d = df.iloc[tr]; d = d[d.avg_position > 0]
    band = lambda p: pd.cut(p, [0, 3, 10, 20], labels=["1-3", "3-10", "10-20"])
    g = d[d.avg_position <= 20].assign(b=lambda t: band(t.avg_position)).groupby("b", observed=True)
    norm = (g.clicks_90d.sum() / g.impressions_90d.sum() * 100)
    t = df.iloc[te]
    ok = (t.avg_position > 0) & (t.avg_position <= 20) & (t.impressions_90d >= 1000)
    exp = band(t.avg_position).map(norm).astype(float)
    return np.where(ok, t.impressions_90d * (exp - t.ctr).clip(lower=0) / 100, 0.0)

MODELS = {
    "logistic regression": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                                 LogisticRegression(max_iter=2000, C=0.5)),
    "decision tree (depth 3)": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                                     DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=SEED)),
    "random forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                           RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED)),
}
KS = [20, 50, 200]
rows, oof = {}, {}
def record(name, scores):
    r = {"AUC": np.mean([roc_auc_score(y[te], scores[i]) if len(set(scores[i])) > 1 else 0.5 for i, (_, te) in enumerate(folds)])}
    for k in KS:
        r[f"P@{k}"] = np.mean([prec_at_k(scores[i], y[te], k) for i, (_, te) in enumerate(folds)])
    rows[name] = r

record("W4 baseline (CTR-gap rule)", [baseline_score(tr, te) for tr, te in folds])
fold_scores = {}
for name, mk in MODELS.items():
    sc = []
    for tr, te in folds:
        m = mk().fit(X.iloc[tr], y[tr]); sc.append(m.predict_proba(X.iloc[te])[:, 1])
    fold_scores[name] = sc; record(name, sc)

table = pd.DataFrame(rows).T
table.loc["(random picking = base rate)"] = [0.5] + [y.mean()] * len(KS)
print(table.round(3).to_string())
json.dump(table.round(4).to_dict("index"), open("work/outputs/w05_model_metrics.json", "w"), indent=2)

                                AUC   P@20   P@50  P@200
W4 baseline (CTR-gap rule)    0.548  0.530  0.560  0.588
logistic regression           0.656  0.760  0.796  0.783
decision tree (depth 3)       0.610  0.610  0.620  0.617
random forest                 0.674  0.770  0.772  0.763
(random picking = base rate)  0.500  0.542  0.542  0.542


**How to read the table:** AUC and P@K are means over the five held-out client folds. Random picking gives AUC 0.5 and precision equal to the base rate. Because the base rate of "declining" is already high (about 54%), a model must clear that bar by a clear margin to be worth using.

In [5]:
tree = MODELS["decision tree (depth 3)"]().fit(X, y)
print(export_text(tree[-1], feature_names=list(X.columns), max_depth=3))

|--- days_with_impressions <= 4.50
|   |--- avg_position <= 0.75
|   |   |--- sessions_90d <= 1.50
|   |   |   |--- class: 0
|   |   |--- sessions_90d >  1.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 111.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  111.50
|   |   |   |--- class: 0
|--- days_with_impressions >  4.50
|   |--- content_age_days <= 344.50
|   |   |--- clicks_90d <= 3.35
|   |   |   |--- class: 1
|   |   |--- clicks_90d >  3.35
|   |   |   |--- class: 0
|   |--- content_age_days >  344.50
|   |   |--- avg_position <= 21.35
|   |   |   |--- class: 1
|   |   |--- avg_position >  21.35
|   |   |   |--- class: 0



## 4. Errors and interpretation

In [6]:
from sklearn.inspection import permutation_importance
best = table.drop(index=["(random picking = base rate)", "W4 baseline (CTR-gap rule)"]).AUC.idxmax()
print("best model by AUC:", best)

# held-out predictions for every page (each page scored by the fold where its client was in test)
oof = np.zeros(len(df))
for i, (tr, te) in enumerate(folds): oof[te] = fold_scores[best][i]
df["p_declining"] = oof

# permutation importance on one held-out fold (AUC drop when a column is shuffled)
tr, te = folds[0]
m = MODELS[best]().fit(X.iloc[tr], y[tr])
pi = permutation_importance(m, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print("\nTop permutation importances (AUC drop, fold 0):"); print(imp.head(8).round(4).to_string())

# where is it wrong? AUC by content type and by client
print("\nAUC by content_type:")
for ct, g in df.groupby("content_type"):
    print(f"  {ct:20s} n={len(g):>6,}  AUC={roc_auc_score(g.declining, g.p_declining):.3f}  base={g.declining.mean():.2f}")
ca = pd.Series({c: roc_auc_score(g.declining, g.p_declining) for c, g in df.groupby("client_id")
                if g.declining.nunique() == 2 and len(g) >= 100})
print(f"\nper-client AUC (clients with >=100 pages, n={len(ca)}): min {ca.min():.2f}, median {ca.median():.2f}, max {ca.max():.2f}")

# 3 concrete confident mistakes
wrong = df[(df.declining == 0)].sort_values("p_declining", ascending=False).head(3)
print("\nThree most confident false alarms (predicted declining, actually not):")
print(wrong[["content_id", "p_declining", "impressions_90d", "avg_position", "days_with_impressions", "trend_direction", "trend_pct"]].round(3).to_string(index=False))

best model by AUC: random forest



Top permutation importances (AUC drop, fold 0):
avg_position             0.0787
content_age_days         0.0329
days_with_impressions    0.0040
clicks_90d               0.0040
impressions_90d          0.0033
has_word_count           0.0017
search_volume            0.0015
engagement_rate          0.0002

AUC by content_type:
  comparison article   n=   697  AUC=0.541  base=0.57
  feedly article       n= 2,096  AUC=0.859  base=0.29
  keyword article      n=27,207  AUC=0.668  base=0.56



per-client AUC (clients with >=100 pages, n=23): min 0.52, median 0.67, max 0.82

Three most confident false alarms (predicted declining, actually not):
          content_id  p_declining  impressions_90d  avg_position  days_with_impressions trend_direction  trend_pct
content_3bb23afaf34e        0.940             2910          15.4                     88          stable        0.5
content_2dbab51b83c9        0.922              781           9.1                     79              up       24.0
content_d015eb800625        0.921             1064           8.5                     75          stable       -0.4


**What the errors look like (numbers from the output above, seed 42):**

- **Model vs baseline.** The Week-4 CTR-gap rule is close to a coin flip for *declining* (AUC 0.55, P@50 0.56 against a 0.54 base rate); that is expected, because it was built to find missed clicks, not decline. Logistic regression (AUC 0.66, P@50 0.80) and Random Forest (AUC 0.67, P@50 0.77) beat it on every metric. The forest wins on AUC; logistic is as good or better at P@20/P@50. Since the readable logistic model nearly matches the forest, I would prefer it for decision-support. The depth-3 tree is weaker (AUC 0.61) but its printed rules are the easiest to read.
- **What it leans on.** Permutation importance is dominated by `avg_position` and `content_age_days`; everything else is tiny. The tree shows why: pages with `avg_position = 0` ("no position data") or very few impression days are almost never labelled declining. That is partly a *measurement-coverage* effect (pages with no search footprint cannot lose 20%), not evidence that rank or age *causes* decline. Treat it as directional.
- **Where it is weaker.** By content type, AUC is 0.86 for `feedly article` (but its base rate is only 0.29, so it is the easy group), 0.67 for `keyword article` (91% of rows) and 0.54 for `comparison article` (697 rows, near chance). Per-client AUC ranges from 0.52 to 0.82 (median 0.67), so the model transfers unevenly to unseen clients.
- **Three false alarms.** The top confident mistakes are pages with 75-88 active impression days at positions 8-15 whose impressions were flat or up (`trend_pct` +0.5, +24.0, -0.4). They look like the declining pages on every feature the snapshot offers, so these errors are hard to avoid with a single 90-day snapshot.
- **Honest limits.** Folds have very different base rates (0.38 to 0.65) and fold 0 is a single large client, so P@K means move by several points between runs; differences smaller than about 0.03 should not be read as real. `impressions_90d` overlaps the label window slightly (section 1). A time-aware split needs the warehouse daily table, which I leave for the next step.

**Decision:** the models beat the baseline and the base rate on the same client-held-out folds, but the gain from the forest over the readable logistic model is small, so complexity is not what earns the result.

## Self-check

- [x] Every section filled: markdown thinking and the code behind it
- [x] Same split (client-grouped 5-fold), same target, same metrics for baseline and models; baseline re-fitted on train folds only
- [x] Method choice explained; features/exclusions listed; label-derived columns excluded
- [x] Metrics: AUC, precision@20/50/200, base rate; permutation importance; per-group errors; 3 wrong cases
- [x] No client names or URLs; careful wording (observed, directional, decision-support)
- [x] Seeds fixed (42); metrics JSON written to `work/outputs/w05_model_metrics.json`